# リソースクリーンアップ
ハンズオンで作成したリソースを削除します。
> **注意**: このnotebookを実行すると、ハンズオンで作成した全てのリソースが削除されます。

In [ ]:
%pip install -U -qqqq databricks-vectorsearch databricks-sdk
dbutils.library.restartPython()

In [ ]:
%run ./config

## Step 1: Vector Search Indexの削除

In [ ]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

# Code スキーマのインデックス
for idx_name in [
    f"{catalog}.{schema_code}.knowledge_base_vs_index",
    f"{catalog}.{schema_nocode}.knowledge_base_vs_index",
]:
    try:
        vsc.delete_index(VECTOR_SEARCH_ENDPOINT_NAME, idx_name)
        print(f"✅ Vector Search Index '{idx_name}' を削除しました")
    except Exception as e:
        print(f"⚠️ Index削除スキップ ({idx_name}): {e}")

## Step 2: Vector Search Endpointの削除

In [ ]:
try:
    vsc.delete_endpoint(VECTOR_SEARCH_ENDPOINT_NAME)
    print(f"✅ Vector Search Endpoint '{VECTOR_SEARCH_ENDPOINT_NAME}' を削除しました")
except Exception as e:
    print(f"⚠️ Endpoint削除スキップ: {e}")

## Step 3: Databricks Appの削除

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
app_name = "ricoh-technova-chatbot"

try:
    w.apps.delete(name=app_name)
    print(f"✅ Databricks App '{app_name}' を削除しました")
except Exception as e:
    print(f"⚠️ App削除スキップ: {e}")

## Step 4: Model Serving Endpointの削除

In [ ]:
try:
    w.serving_endpoints.delete(ENDPOINT_NAME)
    print(f"✅ Serving Endpoint '{ENDPOINT_NAME}' を削除しました")
except Exception as e:
    print(f"⚠️ Serving Endpoint削除スキップ: {e}")

## Step 5: Secret Scopeの削除

In [ ]:
scope_name = "ricoh_handson"
try:
    w.secrets.delete_scope(scope=scope_name)
    print(f"✅ Secret Scope '{scope_name}' を削除しました")
except Exception as e:
    print(f"⚠️ Secret Scope削除スキップ: {e}")

## Step 6: MLflow登録モデルの削除

In [ ]:
import mlflow
mlflow.set_registry_uri("databricks-uc")

# 登録モデルの削除
for schema_name in [schema_code, schema_nocode]:
    try:
        model_full_name = f"{catalog}.{schema_name}.{MODEL_NAME}"
        client = mlflow.MlflowClient()
        client.delete_registered_model(model_full_name)
        print(f"✅ 登録モデル '{model_full_name}' を削除しました")
    except Exception as e:
        print(f"⚠️ モデル削除スキップ ({schema_name}): {e}")

## Step 7: UC関数の削除（Code スキーマ）

In [ ]:
functions_to_drop = [
    "get_customer_by_email",
    "get_order_history",
    "get_support_tickets",
    "search_policy",
    "calculate_math_expression",
]

for func_name in functions_to_drop:
    try:
        spark.sql(f"DROP FUNCTION IF EXISTS {catalog}.{schema_code}.{func_name}")
        print(f"✅ 関数 '{func_name}' を削除しました")
    except Exception as e:
        print(f"⚠️ 関数 '{func_name}' 削除スキップ: {e}")

## Step 8: 両スキーマの削除

In [ ]:
for schema_name in [schema_code, schema_nocode]:
    try:
        spark.sql(f"DROP SCHEMA IF EXISTS {catalog}.{schema_name} CASCADE")
        print(f"✅ スキーマ '{catalog}.{schema_name}' を削除しました")
    except Exception as e:
        print(f"⚠️ スキーマ削除スキップ ({schema_name}): {e}")

## クリーンアップ完了
全てのリソースが削除されました。
> **注意**: AI Gateway (`yao-ai-agent-demo-gateway`) と埋め込みモデル (`databricks-qwen3-embedding-0-6b`) は共有リソースのため削除していません。